# Rvector & RvectorPack — Ring Vectors

Both Rvector and RvectorPack store ring vectors, with each dimension in the ring $\mathbb{Z}_{2^ell}$. Since the current MPMT application scope only requires ring operations in the range 1–256, $\ell \in [1, 8]$. Accordingly, Rvector and RvectorPack use one byte per ring element at the storage layer (or one bit when $\ell=1$). Rvector is the computational representation of ring element vectors, while RvectorPack is the storage representation — a compact form of Rvector. When $\ell=1$ or $\ell=8$, the representation is naturally compact; for other values of $\ell$, each element occupies one byte and must be compressed into RvectorPack form for storage and transport.

## Construction

`Rvector(ell)` returns a **type** (not an instance); calling the type allocates a vector. Vectors are not value-initialized upon allocation; call `.fill(val)` or `.rand_fill()` manually to populate them.

In [2]:
import mpmt

ell=3

# Rv is the Rvector class for ring size ell
Rv = mpmt.Rvector(ell=ell)

# Allocate a vector of size 10 (uninitialized)
v1 = Rv(size=10)
print(v1)

# Fill with an initial value
v1.fill(1)
print(v1)

# Fill value must not exceed the ring modulus
try:
    v1.fill(9)
except ValueError as e:
    print(f"ValueError: {e}")

# Fill with cryptographically secure random values
v1.rand_fill()
print(v1)


RvectorEll3(size=10, [1, 7, 5, 5, 1, 7, 1, 0, ...])
RvectorEll3(size=10, [1, 1, 1, 1, 1, 1, 1, 1, ...])
ValueError: Rvector<3>: value 9 exceeds ring modulus 2^3
RvectorEll3(size=10, [1, 1, 0, 2, 7, 3, 6, 6, ...])


## Element Access

| Operation | Description |
|------|------|
| `v[i]` | Read the `i`-th element; out-of-bounds raises `IndexError` |
| `v[i] = val` | Write the `i`-th element |
| `v.fill(val=0)` | Fill all elements with `val` |
| `v.rand_fill()` | Fill all elements with cryptographically secure random values |
| `v.batch_set(indices, val)` | Bulk write: `v[indices[i]] = val` |
| `v.batch_get(indices, out)` | Bulk read: `out[i] = v[indices[i]]` |

`indices` must be `array("Q")`. For `batch_get`, `out` must be pre-allocated with `out.size >= len(indices)`.

In [3]:
from array import array

Rv = mpmt.Rvector(ell=ell)
v = Rv(10)
v.fill(0)
print(f"initial: {v}")

# Single-element read/write
v[0] = 5
v[1] = 12
print(f"single-element r/w: {v}")

# Bulk write
indices = array("Q", [0, 1, 2])
v.batch_set(indices=indices, val=3)
print(f"bulk r/w: {v}")

# Bulk read
out = Rv(3)
v.batch_get(indices=indices, out=out)
print(f"batch_get: out[0] = {out[0]}, out[1] = {out[1]}, out[2] = {out[2]}")

initial: RvectorEll3(size=10, [0, 0, 0, 0, 0, 0, 0, 0, ...])
single-element r/w: RvectorEll3(size=10, [5, 4, 0, 0, 0, 0, 0, 0, ...])
bulk r/w: RvectorEll3(size=10, [3, 3, 3, 0, 0, 0, 0, 0, ...])
batch_get: out[0] = 3, out[1] = 3, out[2] = 3


## Serialization

| Method | Description |
|------|------|
| `v.to_bytes()` | Export as raw storage bytes (bit-packed representation, no metadata) |
| `v.from_bytes(data)` | Restore from bytes; length must match exactly; calls `canonicalize()` automatically |

In [4]:
v = Rv(10)
v.rand_fill()

data = v.to_bytes()
v2 = Rv(10)
v2.from_bytes(data)
print(f"round-trip: {v == v2}")

# Length mismatch raises ValueError
try:
    v3 = Rv(50)
    v3.from_bytes(data)
except ValueError as e:
    print(f"ValueError: {e}")

round-trip: True
ValueError: from_bytes: expected 50 bytes, got 10


## Arithmetic (Static Methods)

All arithmetic operations are **static methods**; `out` must be pre-allocated and **must not alias** `a` or `b` (the library does not check for aliasing). The ring size is determined by the Rvector.

### Vector-Vector

| Method | Signature | Semantics |
|------|------|------|
| `add` | `(a, b, out)` | `out[i] = a[i] + b[i] mod 2^ELL` |
| `sub` | `(a, b, out)` | `out[i] = a[i] - b[i] mod 2^ELL` |
| `hadamard` | `(a, b, out)` | `out[i] = a[i] · b[i] mod 2^ELL` |

### Vector-Scalar

| Method | Signature | Semantics |
|------|------|------|
| `add_scalar` | `(a, scalar, out)` | `out[i] = a[i] + scalar mod 2^ELL` |
| `sub_scalar` | `(a, scalar, out)` | `out[i] = a[i] - scalar mod 2^ELL` |
| `mul_scalar` | `(a, scalar, out)` | `out[i] = a[i] · scalar mod 2^ELL` |

### Reduction

| Method | Signature | Returns |
|------|------|------|
| `dot` | `(a, b)` | `Σ a[i]·b[i] mod 2^ELL` |
| `reduce` | `(a)` | `Σ a[i] mod 2^ELL` |

In [12]:

len1=10
a = Rv(len1); b = Rv(len1); out = Rv(len1)
a.fill(7)
b.rand_fill()

# Vector-vector
Rv.add(a, b, out)
print(f"add: out = {out}")

Rv.sub(a, b, out)
print(f"sub: out = {out}")

Rv.hadamard(a, b, out)
print(f"hadamard: out = {out}")

# Vector-scalar
Rv.add_scalar(a, 1, out)
print(f"add_scalar: out = {out}")

Rv.sub_scalar(a, 1, out)
print(f"sub_scalar: out = {out}")

Rv.mul_scalar(a, 3, out)
print(f"mul_scalar: out = {out}")

# Reduction
d = Rv.dot(a, out)
s = Rv.reduce(a)
print(f"dot = {d}, reduce = {s}")

add: out = RvectorEll3(size=10, [3, 0, 1, 7, 1, 1, 7, 0, ...])
sub: out = RvectorEll3(size=10, [3, 6, 5, 7, 5, 5, 7, 6, ...])
hadamard: out = RvectorEll3(size=10, [4, 7, 6, 0, 6, 6, 0, 7, ...])
add_scalar: out = RvectorEll3(size=10, [0, 0, 0, 0, 0, 0, 0, 0, ...])
sub_scalar: out = RvectorEll3(size=10, [6, 6, 6, 6, 6, 6, 6, 6, ...])
mul_scalar: out = RvectorEll3(size=10, [5, 5, 5, 5, 5, 5, 5, 5, ...])
dot = 6, reduce = 6


## RvectorPack — Packed Scratch Buffer

`RvectorPack(ell)(n)` — Pre-allocated buffer for file I/O and RingTransport pack/unpack operations.

Given `p = RvectorPack(ell)(n)`, the following attributes and methods are available:

| Attribute | Description |
|------|------|
| `p.size` | Size in bytes |
| `p.ell` | Ring bit-width |
| `p.n_elements` | Number of elements it can hold |
| `p.to_bytes()` | Export as bytes |

Used together with module-level functions `rvector_pack` / `rvector_unpack`:

In [ ]:
import mpmt
plen = 1_000_000

v = Rv(plen)
v.rand_fill()
print(v)

# Pack
buf = mpmt.RvectorPack(ell=ell)(n=plen)
mpmt.rvector_pack(v, buf)

# Unpack
v2 = Rv(plen)
mpmt.rvector_unpack(buf, v2)
print(v2)
print(f"round-trip OK: {v == v2}")

Similarly, file I/O is supported:

`v.save(path, aux_buf)` / `v.load(path, aux_buf)` — Compact bit-packed disk persistence. `path` is the file path, **with the required suffix `.mpmtrvp`**.

Both methods release the GIL and are suitable for multi-threaded use. `load` validates that the on-disk ELL and n match.

In [ ]:
import tempfile, os

vlen = 1_000_000
v = Rv(vlen)
v.rand_fill()

aux = mpmt.RvectorPack(ell=ell)(n=vlen)

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "demo.mpmtrvp")
    v.save(path, aux)

    fsize_mb = os.path.getsize(path) / 1e6
    raw_mb = v.size / 1e6
    print(f"raw={raw_mb:.1f} MB  packed={fsize_mb:.1f} MB  "
          f"ratio={raw_mb/fsize_mb:.1f}:1")

    v2 = Rv(vlen)
    v2.load(path, aux)
    assert v == v2
    print("round-trip: OK")